# Intrinsic Camera / Telescope Split — Z4 (v1)

**Author:** Aaron Roodman
**Date Created:** 2026-06-10
**Last Modified:** 2026-06-10
**Status:** Draft
**Keywords:** AOS, FAM, intrinsic, Z4, defocus, rotator, OCS, CCS, decomposition

## Description

The measured intrinsic Z4 map (after the CCD-height contribution is removed,
`z4_optical_OCS`) appears to contain a component fixed to the **camera** (so it
rotates with the rotator, constant in CCS) on top of a component fixed to the
**telescope** mirrors (constant in OCS).  Using FAM-derived intrinsic maps built
at several rotator angles, this notebook decomposes the map into:

* **O** — telescope-fixed (OCS) map, and
* **C** — camera-fixed (CCS) map that rotates with the rotator.

Model, per OCS field point `(r, phi)` for a dataset at rotator `theta`:

```
Z_theta(r, phi) = O(r, phi) + C(r, phi - s*theta)
```

Rotation is diagonal in the azimuthal Fourier basis, so per `(r, m)` the
coefficients satisfy `A_{d,m} = O_m + C_m * exp(-i m s theta_d)`, a 2-unknown
least-squares solve across the rotator datasets.  The axisymmetric `m=0`
profile is mathematically unsplittable (identical in both frames) and is
assigned by the `m0_assignment` flag.  The decomposition core lives in
`code/intrinsic_split.py`.

**Output:** PDF of data / O / C / residual maps + a parquet of the O (OCS) and
C (CCS) maps.  Files in `output/intrinsic_split/`.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-06-10 | Aaron Roodman | Initial version: Fourier-per-radius decomposition of the Z4-optical intrinsic map into telescope-fixed (OCS) and camera-fixed (CCS) components from FAM maps at 5 rotator angles; auto rotation-sign, m0_assignment flag, actual mean rotator from the fits parquet, diagnostics + maps + output parquet. |
| 2026-06-10 (v2) | Aaron Roodman | Added hole-aware decomposition (decompose_polar_lsq): per-radius masked Fourier least-squares fits O and C from valid samples only, so the telescope map O is hole-free while the camera-dead CCS regions are smoothly filled in C.  Now the default (decomp_method='lsq', hole_dist=0.06 deg).  Verified sign convention OCS<-CCS = +1*rotator from the donut parquet (matches auto s=+1). |
| 2026-06-10 (v3) | Aaron Roodman | Bumped reconstruction r_max to 1.75 deg with denser radial sampling (n_r=80) so O fills to ~1.73 deg (last well-covered radius; the ~empty 1.75 ring is skipped) and added light ridge regularization (ridge=1e-3) so partially-covered outer rings do not ring. Added a detailed markdown explanation of the Fourier-per-radius fit and a diagnostics page (camera azimuthal-mode spectrum, axisymmetric m=0 radial profile, residual-vs-radius). |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Decomposition](#decomp)
6. [Maps & Diagnostics](#plots)
7. [Output](#output)

<a id='params'></a>
## 1. Parameters

In [1]:
# ============================================================
# Parameters
# ============================================================
# Intrinsic grid maps built at different rotator angles (one per dataset).
# Each `subdir` holds `grid_filename`; `rot_range`/`alt_range` (deg) select
# the visits the map was built from (used to read the actual mean rotator).
intrinsic_base = 'output/build_measured_intrinsic/fam_danish_1_0_wep17_3_0_bin2x'
grid_filename  = 'measured_intrinsic_nkeep_34_pathA_grid.parquet'
datasets = [
    {'label': 'rot-60', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_-65.0_-55.0',
     'rot_range': (-65.0, -55.0), 'alt_range': (55.0, 75.0)},
    {'label': 'rot-15', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_-20.0_-10.0',
     'rot_range': (-20.0, -10.0), 'alt_range': (55.0, 75.0)},
    {'label': 'rot00',  'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_-3.0_3.0',
     'rot_range': (-3.0,  3.0),  'alt_range': (55.0, 75.0)},
    {'label': 'rot+15', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_10.0_20.0',
     'rot_range': (10.0, 20.0),  'alt_range': (55.0, 75.0)},
    {'label': 'rot+60', 'subdir': 'nkeep_34_OCS_riz_alt_55.0_75.0_rot_55.0_65.0',
     'rot_range': (55.0, 65.0),  'alt_range': (55.0, 75.0)},
]

# Quantity to decompose (a scalar / rotationally-invariant Zernike).
value_col = 'z4_optical_OCS'

# FAM fits parquet(s) used to read the actual mean rotator per dataset.
fits_parquet_paths = [
    'output/aos_fam_danish_1_0_wep17_3_0_bin2x_20260315_20260327_fits.parquet',
    'output/aos_fam_danish_1_0_wep17_3_0_bin2x_20260331_20260409_fits.parquet',
    'output/aos_fam_danish_1_0_wep17_3_0_bin2x_20260418_20260531_fits.parquet',
]
rotator_angle_source = 'data'    # 'data' (mean from fits parquet) | 'dirname_center'

# Degenerate axisymmetric (m=0) part -> telescope by default (settable).
m0_assignment = 'ocs'            # 'ocs' | 'ccs' | 'split'
# Rotation sense; 'auto' picks the sign with the lower 2-component residual.
rotation_sign = 'auto'           # 'auto' | 1 | -1
# Decomposition method: 'lsq' (hole-aware, telescope map O hole-free) or 'fft'.
decomp_method = 'lsq'
# A polar node whose nearest real grid bin is farther than hole_dist (deg) sits
# in a dead-detector hole and is excluded from the fit (lsq only).  ~1.2 bins.
hole_dist = 0.06
# Tikhonov factor damping azimuthal modes the ring coverage can't
# constrain (keeps partially-covered outer rings from ringing).
ridge = 1e-3
# Azimuthal-order cap (lsq: real unknowns = 1 + 4*m_max per ring; fft: None=all).
m_max = 12

# Polar sampling grid + the radial window used for residual/RMS metrics.
polar = dict(r_min=0.06, r_max=1.75, n_r=80, n_az=180)
r_lim = (0.1, 1.6)

output_dir     = 'output/intrinsic_split'
output_pdf     = f'{output_dir}/intrinsic_split_z4.pdf'
output_parquet = f'{output_dir}/intrinsic_split_z4_maps.parquet'

<a id='setup'></a>
## 2. Setup & Imports

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, 'code')
import intrinsic_split as isp
from common.utils import setup_plotting

setup_plotting()
print('Ready.')

Ready.


<a id='functions'></a>
## 3. Helper Functions

In [3]:
def plot_field(ax, X, Y, vals, title, vlim, levels=21, cmap='RdBu_r'):
    """Filled-contour map of a field sampled on the polar-grid points (X, Y)."""
    xr, yr, vr = X.ravel(), Y.ravel(), np.asarray(vals).ravel()
    fin = np.isfinite(vr)                       # drop NaN (e.g. skipped outer ring)
    im = ax.tricontourf(xr[fin], yr[fin], vr[fin],
                        levels=np.linspace(-vlim, vlim, levels),
                        cmap=cmap, extend='both')
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('thx [deg]', fontsize=8)
    ax.set_ylabel('thy [deg]', fontsize=8)
    ax.tick_params(labelsize=7)
    return im


def vlim_of(Z, R, r_lim, pct=98):
    """Symmetric colour limit from the in-window percentile of |Z|."""
    m = (R >= r_lim[0]) & (R <= r_lim[1])
    return float(np.nanpercentile(np.abs(np.asarray(Z)[..., m, :]), pct))

<a id='data'></a>
## 4. Data Access

Load the intrinsic grid map for each rotator dataset and read the actual mean
rotator angle of the visits each was built from.

In [4]:
fits_table = None
if rotator_angle_source == 'data':
    cols = ['visit', 'rotator_angle', 'alt', 'day_obs', 'seq_num']
    parts = []
    for p in fits_parquet_paths:
        pp = Path(p)
        if pp.exists():
            parts.append(pd.read_parquet(pp, columns=cols))
        else:
            print(f'  WARNING: fits parquet not found: {pp}')
    fits_table = (pd.concat(parts, ignore_index=True).drop_duplicates('visit')
                  if parts else None)
    if fits_table is None:
        print('  No fits parquet available -> falling back to dirname centers.')

maps, thetas, labels = [], [], []
for ds in datasets:
    gp = Path(intrinsic_base) / ds['subdir'] / grid_filename
    if not gp.exists():
        raise FileNotFoundError(gp)
    df = pd.read_parquet(gp)
    f = np.isfinite(df[value_col].values)
    maps.append((df['thx_deg'].values[f], df['thy_deg'].values[f],
                 df[value_col].values[f]))
    center = float(np.mean(ds['rot_range']))
    if fits_table is not None:
        mean_rot, n_v, lo, hi = isp.mean_rotator(
            fits_table, ds['rot_range'], ds['alt_range'])
        if not np.isfinite(mean_rot):
            mean_rot, n_v, lo, hi = center, 0, np.nan, np.nan
    else:
        mean_rot, n_v, lo, hi = center, 0, np.nan, np.nan
    thetas.append(mean_rot)
    labels.append(ds['label'])
    print(f"{ds['label']:7s}: {int(f.sum()):4d} grid pts, "
          f"rotator mean={mean_rot:+.2f} deg (n={n_v}, "
          f"range {lo:+.1f}..{hi:+.1f}, dir-center {center:+.1f})")

thetas = np.array(thetas)
print(f'\nUsing rotator angles (deg): {np.round(thetas, 2)}')

rot-60 : 3856 grid pts, rotator mean=-59.90 deg (n=60, range -60.1..-59.7, dir-center -60.0)
rot-15 : 3874 grid pts, rotator mean=-14.94 deg (n=53, range -16.9..-11.3, dir-center -15.0)
rot00  : 3887 grid pts, rotator mean=-0.27 deg (n=217, range -2.4..+2.2, dir-center +0.0)
rot+15 : 3898 grid pts, rotator mean=+14.43 deg (n=108, range +10.8..+15.0, dir-center +15.0)
rot+60 : 3869 grid pts, rotator mean=+59.61 deg (n=93, range +59.2..+60.1, dir-center +60.0)

Using rotator angles (deg): [-59.9  -14.94  -0.27  14.43  59.61]


<a id='decomp'></a>
## 5. Decomposition

Sample every map onto a common polar grid, then solve `A_{d,m} = O_m +
C_m e^{-i m s theta_d}` per `(r, m)`.  The rotation sign is chosen
automatically (lower residual) unless fixed in the parameters.

### How the Fourier-based fit works

Each rotator dataset *d* (rotator angle $\theta_d$) provides the measured map in
**OCS** field coordinates; the camera-fixed part rotates rigidly with the
rotator, so

$$Z_{\theta}(r,\phi) \;=\; O(r,\phi)\;+\;C\!\left(r,\;\phi - s\,\theta\right),$$

where $O$ is the telescope-fixed (OCS) map, $C$ the camera-fixed (CCS) map, and
$s$ the rotation sense (here $s=+1$: from the donut parquet, $\mathrm{OCS}=R(+\,\mathrm{rotator})\,\mathrm{CCS}$).

**Why Fourier in azimuth.** At a fixed radius $r$, a rigid rotation by $\alpha$ in
the azimuth $\phi$ is *diagonal* in the azimuthal Fourier basis $e^{im\phi}$: it
multiplies mode $m$ by $e^{-im\alpha}$.  Expanding
$O(r,\phi)=\sum_m O_m(r)\,e^{im\phi}$ and $C(r,\psi)=\sum_m C_m(r)\,e^{im\psi}$,
the measured coefficients become

$$A_{d,m}(r)\;=\;O_m(r)\;+\;C_m(r)\,e^{-i m s\,\theta_d}.$$

So for each $(r,m)$ there is a **two-unknown linear system** in $(O_m,C_m)$ with
design rows $[\,1,\;e^{-i m s\theta_d}\,]$, one row per dataset.  With $\ge 2$
distinct rotator angles (and $m\neq0$) it is solved by least squares; the 5
angles here overdetermine it, and the residual measures how well a pure
*telescope + rotating-camera* model explains the rotator dependence.

**The $m=0$ degeneracy.** For $m=0$, $e^{-i0\theta}=1$ for every dataset, so only
the **sum** $O_0+C_0$ is determined — an axisymmetric pattern is identical in
both frames.  This is the "can't separate at the field centre" ambiguity,
generalised to the entire axisymmetric radial profile.  The `m0_assignment`
flag assigns it to $O$ (default), to $C$, or splits it 50/50; it changes only
that axisymmetric part, never the residual.

**Two implementations.**
* `fft` (`decompose_polar`) — azimuthal FFT per ring, then the per-$(r,m)$ solve.
  Fast, but assumes full hole-free rings (it interpolates across any holes).
* `lsq` (`decompose_polar_lsq`, **default**) — per ring, fit $O$ and $C$ jointly
  by real-basis least squares (a constant plus $\cos m\phi,\sin m\phi$ up to
  `m_max`) over **only the valid, non-hole samples**.  Because the dead-detector
  holes are camera-fixed and rotate through OCS, every OCS azimuth is sampled by
  *some* dataset, so $O$ is reconstructed **hole-free**; the camera-dead CCS
  regions — never sampled at any rotator — are filled by the smooth Fourier
  reconstruction of $C$.  `m_max` caps the azimuthal order ($1+4\,$`m_max` real
  unknowns per ring); rings with too few valid samples are skipped (left NaN).

**Reconstruction.** $O(r,\phi)$ is evaluated on the OCS grid and $C(r,\psi)$ on
the CCS grid (the same field radii/azimuth, interpreted as CCS).  For dataset
$d$ the predicted OCS map is $O(\phi)+C(\phi - s\theta_d)$.

In [5]:
R, A, X, Y = isp.make_polar_grid(**polar)
Z, valid, n_nan = isp.sample_maps_polar(maps, X, Y, hole_dist=hole_dist)
print(f'Polar samples: {Z.shape}  (NaNs outside hull: {n_nan})')
inwin = (R >= r_lim[0]) & (R <= r_lim[1])
nwin = int(inwin.sum()) * Z.shape[2]
for i, lab in enumerate(labels):
    nbad = int((~valid[i, inwin, :]).sum())
    print(f'  {lab:7s}: {nbad:4d}/{nwin} in-window polar nodes in a hole '
          f'({100 * nbad / nwin:.1f}%)')

th_rad = np.deg2rad(thetas)
if rotation_sign == 'auto':
    result, sign_rms = isp.decompose_auto_sign(
        Z, th_rad, A, R, r_lim=r_lim, m0_assignment=m0_assignment,
        m_max=m_max, valid=valid, method=decomp_method, ridge=ridge)
    print(f"Rotation-sign residual RMS:  s=+1: {sign_rms[+1]:.4f}  "
          f"s=-1: {sign_rms[-1]:.4f}  -> chose s={result['s']:+d}")
elif decomp_method == 'lsq':
    result = isp.decompose_polar_lsq(Z, valid, th_rad, A, s=int(rotation_sign),
                                     m0_assignment=m0_assignment,
                                     m_max=(12 if m_max is None else m_max),
                                     ridge=ridge)
    print(f"Using fixed s={result['s']:+d}  (lsq)")
else:
    result = isp.decompose_polar(Z, th_rad, A, s=int(rotation_sign),
                                 m0_assignment=m0_assignment, m_max=m_max)
    print(f"Using fixed s={result['s']:+d}  (fft)")

O_pol, C_pol, res = result['O_pol'], result['C_pol'], result['res']

# Diagnostics (NaN-aware: masked holes excluded)
data_rms = isp.residual_rms(Z, R, r_lim)
res_rms  = isp.residual_rms(res, R, r_lim)
ev       = isp.explained_variance(Z, res, R, r_lim)
print(f'\nmethod={decomp_method}   data RMS = {data_rms:.4f}   '
      f'residual RMS = {res_rms:.4f}   explained variance = {ev:.2f}')
print(f"|O| RMS = {isp.residual_rms(O_pol[None], R, r_lim):.4f}   "
      f"|C| RMS = {isp.residual_rms(C_pol[None], R, r_lim):.4f}  "
      f"(m0 -> {result['m0_assignment']})")
print(f"O hole-free in-window? NaNs in O = {int(np.isnan(O_pol[inwin]).sum())}")
print('\nper-dataset residual:')
for i, lab in enumerate(labels):
    print(f"  {lab:7s} (rot={thetas[i]:+.1f}): dataRMS="
          f"{isp.residual_rms(Z[i:i+1], R, r_lim):.4f}  "
          f"resRMS={isp.residual_rms(res[i:i+1], R, r_lim):.4f}")
amp = isp.azimuthal_amplitude(C_pol, R, r_lim, n_m=9)
print(f'\nCamera map C azimuthal-mode amplitude m=0..8: {np.round(amp, 3)}')

Polar samples: (5, 80, 180)  (NaNs outside hull: 1029)
  rot-60 :   18/12600 in-window polar nodes in a hole (0.1%)
  rot-15 :   12/12600 in-window polar nodes in a hole (0.1%)
  rot00  :   10/12600 in-window polar nodes in a hole (0.1%)
  rot+15 :    5/12600 in-window polar nodes in a hole (0.0%)
  rot+60 :   12/12600 in-window polar nodes in a hole (0.1%)
Rotation-sign residual RMS:  s=+1: 0.0191  s=-1: 0.0268  -> chose s=+1

method=lsq   data RMS = 0.0462   residual RMS = 0.0191   explained variance = 0.83
|O| RMS = 0.0361   |C| RMS = 0.0219  (m0 -> ocs)
O hole-free in-window? NaNs in O = 0

per-dataset residual:
  rot-60  (rot=-59.9): dataRMS=0.0442  resRMS=0.0116
  rot-15  (rot=-14.9): dataRMS=0.0572  resRMS=0.0316
  rot00   (rot=-0.3): dataRMS=0.0410  resRMS=0.0171
  rot+15  (rot=+14.4): dataRMS=0.0437  resRMS=0.0113
  rot+60  (rot=+59.6): dataRMS=0.0428  resRMS=0.0162

Camera map C azimuthal-mode amplitude m=0..8: [0.    0.795 2.218 0.726 0.671 0.57  0.422 0.41  0.344]


<a id='plots'></a>
## 6. Maps & Diagnostics

Per-rotator data maps, the recovered telescope (O) and camera (C) maps, the
per-dataset residuals, and the effect of the `m0_assignment` choice.

In [6]:
Path(output_dir).mkdir(parents=True, exist_ok=True)
vlim = vlim_of(Z, R, r_lim)
n_pages = 0
with PdfPages(output_pdf) as pdf:
    # Page 1: data maps + O + C + residual
    fig, axs = plt.subplots(2, 4, figsize=(20, 10), layout='constrained')
    for i, lab in enumerate(labels):
        im = plot_field(axs.flat[i], X, Y, Z[i],
                        f'data {lab}  (rot={thetas[i]:+.1f})', vlim)
    plot_field(axs.flat[5], X, Y, O_pol, 'O = telescope (OCS)', vlim)
    plot_field(axs.flat[6], X, Y, C_pol, 'C = camera (CCS)', vlim)
    _ri = len(labels) // 2; _m = np.isfinite(res[_ri])
    im = plot_field(axs.flat[7], X[_m], Y[_m], res[_ri][_m],
                    f'residual {labels[_ri]}', vlim)
    fig.colorbar(im, ax=axs.ravel().tolist(), shrink=0.5,
                 label=f'{value_col} [um]')
    fig.suptitle(f'{value_col}: telescope (OCS) + camera (CCS) split  '
                 f"(s={result['s']:+d}, m0->{result['m0_assignment']})",
                 fontsize=13)
    pdf.savefig(fig, bbox_inches='tight'); plt.show(); plt.close(fig); n_pages += 1

    # Page 2: per-dataset residual maps
    fig, axs = plt.subplots(1, len(labels), figsize=(4.2 * len(labels), 4.2),
                            layout='constrained')
    for i, lab in enumerate(labels):
        _m = np.isfinite(res[i])
        im = plot_field(axs[i], X[_m], Y[_m], res[i][_m],
                        f'residual {lab}', vlim)
    fig.colorbar(im, ax=list(axs), shrink=0.7, label='residual [um]')
    fig.suptitle('Per-dataset residual (data - [O + rotated C])', fontsize=13)
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig); n_pages += 1

    # Page 3: m0_assignment comparison (O and C for each choice)
    fig, axs = plt.subplots(2, 3, figsize=(16, 10), layout='constrained')
    for j, flag in enumerate(('ocs', 'ccs', 'split')):
        rr = (isp.decompose_polar_lsq(Z, valid, th_rad, A, s=result['s'],
                                      m0_assignment=flag,
                                      m_max=(12 if m_max is None else m_max),
                                      ridge=ridge)
              if decomp_method == 'lsq' else
              isp.decompose_polar(Z, th_rad, A, s=result['s'],
                                  m0_assignment=flag, m_max=m_max))
        plot_field(axs[0, j], X, Y, rr['O_pol'], f'O  (m0->{flag})', vlim)
        plot_field(axs[1, j], X, Y, rr['C_pol'], f'C  (m0->{flag})', vlim)
    fig.suptitle('Effect of the axisymmetric (m=0) assignment on O and C',
                 fontsize=13)
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig); n_pages += 1
    # Page 4: diagnostics — camera azimuthal spectrum, axisymmetric
    # profile, residual vs radius.
    fig, ax = plt.subplots(1, 3, figsize=(18, 4.8), layout='constrained')
    amp = isp.azimuthal_amplitude(C_pol, R, r_lim, n_m=10)
    ax[0].bar(range(10), amp, color='steelblue')
    ax[0].set_xlabel('azimuthal mode m'); ax[0].set_ylabel('amplitude')
    ax[0].set_title('C (camera) azimuthal-mode amplitude')
    cnt = np.isfinite(Z).sum(0)
    Zm = np.where(cnt > 0, np.nansum(Z, 0) / np.maximum(cnt, 1), np.nan)
    m0prof = np.nanmean(Zm, axis=1)
    ax[1].plot(R, m0prof, 'o-', ms=4)
    ax[1].axhline(0, color='gray', lw=0.5)
    ax[1].set_xlabel('field radius [deg]'); ax[1].set_ylabel(f'{value_col} [um]')
    ax[1].set_title(f"axisymmetric m=0 profile  (-> {result['m0_assignment']})")
    for i, lab in enumerate(labels):
        rp = np.sqrt(np.nanmean(res[i] ** 2, axis=1))
        ax[2].plot(R, rp, label=f'{lab} ({thetas[i]:+.0f})')
    ax[2].set_xlabel('field radius [deg]'); ax[2].set_ylabel('residual RMS [um]')
    ax[2].set_title('residual RMS vs radius'); ax[2].legend(fontsize=8)
    fig.suptitle('Diagnostics: camera azimuthal spectrum / axisymmetric '
                 'profile / residual vs radius', fontsize=12)
    pdf.savefig(fig, bbox_inches='tight'); plt.close(fig); n_pages += 1
print(f'Wrote {n_pages} pages to {output_pdf}')

/var/folders/7z/rlsxn7l17_lcfvvjqws8_0ywzz2g7p/T/ipykernel_72142/3553014198.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  pdf.savefig(fig, bbox_inches='tight'); plt.show(); plt.close(fig); n_pages += 1


Wrote 4 pages to output/intrinsic_split/intrinsic_split_z4.pdf


/var/folders/7z/rlsxn7l17_lcfvvjqws8_0ywzz2g7p/T/ipykernel_72142/3553014198.py:63: RuntimeWarning: Mean of empty slice
  rp = np.sqrt(np.nanmean(res[i] ** 2, axis=1))


<a id='output'></a>
## 7. Output

Resample O (OCS) and C (CCS) onto the rot=0 dataset's focal-plane grid and
write a parquet.  `O_<col>` is the telescope map at each OCS field point;
`C_<col>` is the camera map at each CCS field point (same field positions).

In [7]:
# Target grid = the rot~0 dataset's field points.
i0 = int(np.argmin(np.abs(thetas)))
thx0, thy0, _ = maps[i0]
O_pts = isp.polar_field_to_points(O_pol, X, Y, thx0, thy0)
C_pts = isp.polar_field_to_points(C_pol, X, Y, thx0, thy0)
out = pd.DataFrame({
    'thx_deg': thx0, 'thy_deg': thy0,
    f'O_{value_col}': O_pts, f'C_{value_col}': C_pts,
})
out.attrs['rotation_sign'] = result['s']
out.attrs['m0_assignment'] = result['m0_assignment']
out.attrs['thetas_deg'] = list(np.round(thetas, 3))
out.attrs['value_col'] = value_col
Path(output_dir).mkdir(parents=True, exist_ok=True)
out.to_parquet(output_parquet)
print(f'Saved {output_parquet}: {len(out)} rows x {len(out.columns)} cols')
print(out.head())

Saved output/intrinsic_split/intrinsic_split_z4_maps.parquet: 3887 rows x 4 cols
    thx_deg   thy_deg  O_z4_optical_OCS  C_z4_optical_OCS
0 -0.295890 -1.726027               NaN               NaN
1 -0.246575 -1.726027               NaN               NaN
2 -0.197260 -1.726027               NaN               NaN
3 -0.147945 -1.726027               NaN               NaN
4 -0.098630 -1.726027               NaN               NaN
